# Scanner de Fingerprint de Geradores (multi-sonda)

**Objetivo:** *ver e quantificar o que estamos atacando* — os resíduos que cada gerador deixa — e rankear, **sem treinar CNN**, quais transformações de imagem mais os destroem.

O ArtiFact dá **8 geradores fake** (GANs e difusão) + 3 fontes reais.

**Mudança em relação à v1:** medir só por espectro é cego pra cor e textura. Agora usamos **3 sondas** (3 formas independentes de medir):

1. **Espectral** — perfil radial de potência (FFT). Pega a digital de alta-frequência (upsampling de GAN).
2. **Cor** — momentos por canal, correlação entre canais, saturação, histogramas. Pega vieses de cor.
3. **Resíduo local** — co-ocorrência de um resíduo passa-alta (vizinhança de pixels). Pega a "impressão" textural/PRNU.

Para cada sonda, um **probe linear** mede a AUC real-vs-fake. **Menor AUC depois da transformação = mais fingerprint destruído** *naquela* forma de medir.

> Pool de transformações expandido (low-pass + branqueamento espectral + cor); `median` removido (lento). Reusa `aug_utils`. Roda em minutos.

In [ ]:
import sys, json, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold

# reusa as MESMAS transformacoes do treino (aug_utils, em ../notebooks_140k)
sys.path.insert(0, str(Path.cwd().parent / "notebooks_140k"))
from aug_utils import apply_preprocess

PROJECT_ROOT = Path.cwd().resolve().parent
_envf = PROJECT_ROOT / "data_root.env"
DATA_ROOT = Path(_envf.read_text().strip()) if _envf.exists() else PROJECT_ROOT / "data"

ARTIFACT_DIR = DATA_ROOT / "raw" / "artifact_faces"
RESULTS_DIR  = PROJECT_ROOT / "artifacts" / "fingerprint_scan"
FIGS_DIR     = PROJECT_ROOT / "reports" / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGS_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE  = 256     # grade comum para FFT
N_REAL    = 200     # imagens reais (balanceadas entre as fontes reais)
N_PER_GEN = 80      # imagens por gerador fake
SEED      = 42
rng = np.random.default_rng(SEED)

print("DATA_ROOT:", DATA_ROOT)
print("ArtiFact existe:", ARTIFACT_DIR.exists())

## 1. Carrega amostras (reais + cada gerador)

A fonte de cada imagem vem do prefixo do nome (`fonte__arquivo.jpg`), gravado pela preparação do ArtiFact (01a).

In [ ]:
def list_by_source(folder):
    groups = {}
    for p in folder.glob("*.*"):
        src = p.name.split("__")[0] if "__" in p.name else "desconhecido"
        groups.setdefault(src, []).append(p)
    return {k: sorted(v) for k, v in groups.items()}

real_groups = list_by_source(ARTIFACT_DIR / "real")
fake_groups = list_by_source(ARTIFACT_DIR / "fake")
REAL_SOURCES = sorted(real_groups)
FAKE_SOURCES = sorted(fake_groups)
print("Reais:", {k: len(v) for k, v in real_groups.items()})
print("Fakes:", {k: len(v) for k, v in fake_groups.items()})

def load_rgb(path):
    return Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)

def sample(files, n):
    if n >= len(files):
        return list(files)
    return [files[i] for i in rng.choice(len(files), size=n, replace=False)]

per_real = max(1, N_REAL // len(REAL_SOURCES))
real_files = []
for s in REAL_SOURCES:
    real_files += sample(real_groups[s], per_real)
real_files = real_files[:N_REAL]
real_pils = [load_rgb(p) for p in real_files]
fake_pils = {s: [load_rgb(p) for p in sample(fake_groups[s], N_PER_GEN)] for s in FAKE_SOURCES}

real_gray = np.stack([np.asarray(p.convert("L"), dtype=np.float32) / 255.0 for p in real_pils])
fake_gray = {s: np.stack([np.asarray(p.convert("L"), dtype=np.float32) / 255.0 for p in pils]) for s, pils in fake_pils.items()}
print("real:", real_gray.shape, "| total fake:", sum(len(v) for v in fake_pils.values()))

## 2. Visualiza o fingerprint espectral

**Resíduo espectral** = espectro médio (FFT 2D, log) do fake menos o do real. Picos/grades = assinatura periódica do gerador (artefatos de upsampling). É *uma* das coisas que queremos destruir.

In [ ]:
def avg_logspectrum(imgs):
    acc = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float64)
    for im in imgs:
        acc += np.fft.fftshift(np.abs(np.fft.fft2(im)))
    return np.log1p(acc / len(imgs))

real_spec = avg_logspectrum(real_gray)
ncol = 4
nrow = math.ceil(len(FAKE_SOURCES) / ncol)
fig, axes = plt.subplots(nrow, ncol, figsize=(3.2 * ncol, 3.2 * nrow))
for ax, g in zip(axes.ravel(), FAKE_SOURCES):
    resid = avg_logspectrum(fake_gray[g]) - real_spec
    lim = resid.std() * 3
    ax.imshow(resid, cmap="seismic", vmin=-lim, vmax=lim)
    ax.set_title(g, fontsize=9); ax.axis("off")
for ax in axes.ravel()[len(FAKE_SOURCES):]:
    ax.axis("off")
plt.suptitle("Residuo espectral (fake - real): picos/grades = fingerprint do gerador", y=1.0)
plt.tight_layout()
plt.savefig(FIGS_DIR / "scan_residuo_espectral.png", dpi=130, bbox_inches="tight")
plt.show()

In [ ]:
def radial_profile(img):
    power = np.fft.fftshift(np.abs(np.fft.fft2(img))) ** 2
    cy, cx = IMG_SIZE // 2, IMG_SIZE // 2
    y, x = np.indices((IMG_SIZE, IMG_SIZE))
    r = np.hypot(x - cx, y - cy).astype(int)
    tbin = np.bincount(r.ravel(), power.ravel())
    nr = np.bincount(r.ravel())
    return (tbin / np.maximum(nr, 1))[:IMG_SIZE // 2]

freqs = np.arange(IMG_SIZE // 2)
plt.figure(figsize=(9, 5))
plt.semilogy(freqs, np.mean([radial_profile(im) for im in real_gray], axis=0), color="black", lw=2.5, label="real")
for g in FAKE_SOURCES:
    plt.semilogy(freqs, np.mean([radial_profile(im) for im in fake_gray[g]], axis=0), lw=1.0, label=g)
plt.xlabel("frequencia radial (0 = baixa ... 128 = alta / Nyquist)")
plt.ylabel("potencia media (log)")
plt.title("Perfil radial de potencia: real vs cada gerador")
plt.legend(fontsize=8, ncol=2); plt.grid(alpha=0.3)
plt.savefig(FIGS_DIR / "scan_perfil_radial.png", dpi=130, bbox_inches="tight")
plt.show()

## 3. Três formas de medir a imagem (as sondas)

Cada sonda extrai um vetor de features por imagem, atacando uma família de resíduo diferente:

- **`spectral`** — perfil radial de potência (log). Digital de frequência.
- **`color`** — média/desvio/assimetria/curtose por canal, correlação entre canais, saturação, histogramas. Vieses de cor.
- **`residual`** — co-ocorrência de um resíduo passa-alta (vizinhança de pixels) + histograma do resíduo. Impressão textural/local.

Um probe linear (regressão logística + CV) mede a AUC real-vs-fake de cada sonda. AUC perto de **1.0** = atalho trivial naquela forma de medir.

In [ ]:
def _box3(a):
    p = np.pad(a, 1, mode="reflect")
    return (p[:-2, :-2] + p[:-2, 1:-1] + p[:-2, 2:] +
            p[1:-1, :-2] + p[1:-1, 1:-1] + p[1:-1, 2:] +
            p[2:, :-2] + p[2:, 1:-1] + p[2:, 2:]) / 9.0

def feat_spectral(gray):
    return np.log1p(radial_profile(gray))

def feat_color(rgb):
    f = []
    for c in range(3):
        ch = rgb[:, :, c].ravel()
        m, s = ch.mean(), ch.std() + 1e-6
        f += [m, s, float(((ch - m) ** 3).mean() / s ** 3), float(((ch - m) ** 4).mean() / s ** 4)]
    R, G, B = rgb[:, :, 0].ravel(), rgb[:, :, 1].ravel(), rgb[:, :, 2].ravel()
    f += [np.corrcoef(R, G)[0, 1], np.corrcoef(R, B)[0, 1], np.corrcoef(G, B)[0, 1]]
    mx, mn = rgb.max(2), rgb.min(2)
    sat = (mx - mn) / (mx + 1e-6)
    f += [sat.mean(), sat.std()]
    for c in range(3):
        h, _ = np.histogram(rgb[:, :, c], bins=8, range=(0, 1), density=True)
        f += h.tolist()
    return np.nan_to_num(np.array(f, dtype=np.float32))

def feat_residual(gray):
    g = gray * 255.0
    resid = g - _box3(g)
    q = (np.clip(np.round(resid), -3, 3) + 3).astype(int)   # 7 niveis 0..6
    h, v = q[:, :-1].ravel(), q[:, 1:].ravel()
    co = np.zeros((7, 7)); np.add.at(co, (h, v), 1); co /= max(co.sum(), 1)
    hist, _ = np.histogram(resid, bins=16, range=(-8, 8), density=True)
    return np.nan_to_num(np.concatenate([co.ravel(), hist, [resid.std()]]).astype(np.float32))

PROBES = ["spectral", "color", "residual"]

def all_feats(pil):
    rgb = np.asarray(pil.convert("RGB"), dtype=np.float32) / 255.0
    gray = np.asarray(pil.convert("L"), dtype=np.float32) / 255.0
    return {"spectral": feat_spectral(gray), "color": feat_color(rgb), "residual": feat_residual(gray)}

def extract(pils):
    fs = [all_feats(p) for p in pils]
    return {pr: np.stack([f[pr] for f in fs]) for pr in PROBES}

def probe_auc(Xreal, Xfake):
    r = np.random.default_rng(0)
    n = min(len(Xreal), len(Xfake))
    X = np.nan_to_num(np.concatenate([Xreal[r.choice(len(Xreal), n, replace=False)],
                                      Xfake[r.choice(len(Xfake), n, replace=False)]]))
    y = np.r_[np.zeros(n), np.ones(n)]
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    return cross_val_score(clf, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), scoring="roc_auc").mean()

print("sondas definidas:", PROBES)

In [ ]:
# Baseline: separabilidade real-vs-fake por sonda, gerador a gerador
Fr = extract(real_pils)
Fg = {g: extract(fake_pils[g]) for g in FAKE_SOURCES}

rows = []
for g in FAKE_SOURCES:
    rows.append({"gerador": g, **{pr: round(probe_auc(Fr[pr], Fg[g][pr]), 3) for pr in PROBES}})
Fg_all = {pr: np.concatenate([Fg[g][pr] for g in FAKE_SOURCES]) for pr in PROBES}
rows.append({"gerador": "POOLED", **{pr: round(probe_auc(Fr[pr], Fg_all[pr]), 3) for pr in PROBES}})
base_df = pd.DataFrame(rows)
baseline = {pr: float(base_df.loc[base_df.gerador == "POOLED", pr].values[0]) for pr in PROBES}

print("Separabilidade real-vs-fake por SONDA (AUC; 1.0 = atalho trivial):\n")
print(base_df.to_string(index=False))
print("\nLeitura: cada gerador tem um fingerprint mais forte em sondas diferentes.")

## 4. Scanner — varredura de transformações (x 3 sondas)

Para cada transformação x intensidade, aplicamos a todas as imagens e re-medimos a AUC pooled **em cada sonda**. Assim vemos *qual resíduo cada método ataca*: um método de cor deve derrubar a sonda `color` mas não a `spectral`, e vice-versa.

> Pode levar alguns minutos (milhares de transformações). Diminua `N_REAL`/`N_PER_GEN` para acelerar.

In [ ]:
SCAN = {
    # low-pass / espectral
    "jpeg":        [80, 60, 40],
    "blur":        [1.0, 2.0, 3.0],
    "downscale":   [1.5, 2.0, 3.0],
    "noise":       [0.04, 0.08],
    # frequencia cirurgica
    "whiten":      [0.5, 0.8, 1.0],
    # cor / canais
    "grayscale":   [1.0],
    "chanshuffle": [1],
    "posterize":   [4, 2],
    "histeq":      [1],
    "gamma":       [0.5, 1.8],
    "saturation":  [0.5, 2.0],
}

def pooled_after(method, value):
    Fr_t = extract([apply_preprocess(p, method, value) for p in real_pils])
    acc = {pr: [] for pr in PROBES}
    for g in FAKE_SOURCES:
        ft = extract([apply_preprocess(p, method, value) for p in fake_pils[g]])
        for pr in PROBES:
            acc[pr].append(ft[pr])
    Ff_t = {pr: np.concatenate(acc[pr]) for pr in PROBES}
    return {pr: probe_auc(Fr_t[pr], Ff_t[pr]) for pr in PROBES}

results = []
print(f"baseline  spectral={baseline['spectral']:.3f}  color={baseline['color']:.3f}  residual={baseline['residual']:.3f}\n")
for method, values in SCAN.items():
    for v in values:
        a = pooled_after(method, v)
        results.append({"method": method, "value": v, **{pr: round(a[pr], 4) for pr in PROBES}})
        print(f"  {method:11s} {str(v):>4}   spectral={a['spectral']:.3f}  color={a['color']:.3f}  residual={a['residual']:.3f}")

(RESULTS_DIR / "scan_multiprobe.json").write_text(json.dumps(
    {"baseline": baseline, "per_generator": rows, "results": results,
     "img_size": IMG_SIZE, "n_real": N_REAL, "n_per_gen": N_PER_GEN}, indent=2))
print("\nsalvo:", RESULTS_DIR / "scan_multiprobe.json")

In [ ]:
# Curvas em U por sonda + heatmap metodo x sonda
df = pd.DataFrame(results)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
for ax, pr in zip(axes, PROBES):
    for method in SCAN:
        pts = [(r["value"], r[pr]) for r in results if r["method"] == method]
        ax.plot(range(len(pts)), [p[1] for p in pts], marker="o", ms=4, label=method)
    ax.axhline(baseline[pr], color="gray", ls="--", lw=1)
    ax.axhline(0.5, color="red", ls=":", lw=1)
    ax.set_title(f"sonda: {pr}  (baseline {baseline[pr]:.3f})")
    ax.set_xlabel("intensidade ->"); ax.set_ylim(0.45, 1.02); ax.grid(alpha=0.3)
axes[0].set_ylabel("AUC  (menor = mais fingerprint destruido)")
axes[-1].legend(fontsize=7, ncol=2, loc="lower left")
plt.suptitle("Scanner multi-sonda: efeito de cada transformacao nas 3 formas de medir")
plt.tight_layout()
plt.savefig(FIGS_DIR / "scan_multiprobe_ucurvas.png", dpi=130, bbox_inches="tight")
plt.show()

# melhor (menor) AUC que cada metodo atinge, por sonda
agg = df.groupby("method")[PROBES].min()
agg["media_3sondas"] = agg.mean(axis=1)
agg = agg.sort_values("media_3sondas")

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(agg[PROBES].values, cmap="RdYlGn_r", vmin=0.5, vmax=1.0, aspect="auto")
ax.set_xticks(range(len(PROBES))); ax.set_xticklabels(PROBES)
ax.set_yticks(range(len(agg))); ax.set_yticklabels(agg.index)
for i in range(len(agg)):
    for j, pr in enumerate(PROBES):
        ax.text(j, i, f"{agg[pr].values[i]:.2f}", ha="center", va="center", fontsize=8)
ax.set_title("Menor AUC por metodo x sonda (verde = destruiu o fingerprint)")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig(FIGS_DIR / "scan_heatmap.png", dpi=130, bbox_inches="tight")
plt.show()

print("Ranking por reducao media nas 3 sondas:\n")
print(agg.round(3).to_string())

## 5. Leitura e próximos passos

**O que mudou em relação à v1 (só espectro):**
- Agora vemos *que tipo* de resíduo cada método ataca. Espera-se que `blur`/`downscale`/`jpeg`/`whiten` derrubem a sonda **espectral**; `grayscale`/`chanshuffle`/`saturation` derrubem a sonda **cor**; e o resíduo passa-alta capture o que o `noise` e os low-pass fazem na textura local.
- A **tabela por gerador** (seção 3) mostra que GANs e difusão deixam fingerprints em sondas diferentes — por isso um pool diverso importa.

**Como interpretar (honesto):**
- O alvo **não** é minimizar AUC a qualquer custo — derrubar até 0.5 pode significar destruir o sinal todo. Queremos colapsar o atalho **específico do gerador** mantendo o sinal compartilhado.
- Um bom candidato a augmentation ataca **mais de uma sonda** sem zerar nenhuma — ou um *combo* (encadear um método espectral + um de cor) cobre famílias que um método só não cobre.
- Continua um **proxy**: a confirmação real é treinar um CNN com o candidato como augmentation e medir cross-generator (**leave-one-generator-out**).

**Próximo passo:** levar os top candidatos (e 1-2 combos) para o treino leave-one-generator-out, e usar este pipeline de sondas como base do **classificador de geradores** adversarial.